# Hackathon project

The whole project, top to bottom: problem, data, baseline, model, evaluation, demo.
Run it with **Restart kernel and run all**. It must pass at every moment.

In [1]:
# Setup: pick up changes to src/ without restarting, and make `import src` work from notebooks/
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "..")
import src

In [2]:
# One seed for everything random, so every run gives the same numbers
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("seed:", SEED)

seed: 42


## Problem
**Who it is for:** the owner of a small UK supplier, deciding whether to give a young company 30 days of credit.

**What goes in and what comes out:** a customer's company number goes in. FLAG, CLEAR or NOT ENOUGH EVIDENCE comes out, with the reason.

**Why it is hard:** after a company goes insolvent, the same people can start again in a new one (a "phoenix"). A matching name is not proof. A common name raises false alarms, and a new officer id on the register hides the real trail.

This project is a prototype and is not financial advice.

> **Where this notebook stands.** Every box below runs on the *tiny world*: 28 hand-made rows with invented names and known answers. That checks that the pipeline works. It is **not a result**. The real Companies House data comes with card #36, and the final numbers with card #45.

## Data
For now the data is the tiny world (`tests/tiny_world.py`). It comes as the same two tables the real data will: **mentions** (one row per company or person, as one record describes them) and **claims** (one row per "this person is a director of this company", with its dates).

In [3]:
import pandas as pd
from src import evidence as ev
from tests.tiny_world import tiny_world

# truth holds the known answers. It checks the pipeline, and it makes the stand-in probabilities
# in the Model section. No real method ever reads it.
mentions, claims, truth = tiny_world()
ev.check_rows(mentions, claims)    # stops here if a row breaks the shared schema
queries = truth["queries"]         # the five companies a supplier asks about

print(mentions["kind"].value_counts().to_dict(), "and", len(claims), "claims")
mentions.loc[mentions["kind"] == "company", ["company_number", "display_name", "incorporated_on", "insolvency_on"]]

{'company': 9, 'person': 9} and 10 claims


,company_number,display_name,incorporated_on,insolvency_on
0,TW000001,Harrow Build Ltd,2017-02-01,2021-03-15
1,TW000002,Harrow Build Services Ltd,2021-04-20,2022-09-01
2,TW000003,Northgate Roofing Ltd,2021-06-01,NaN
3,TW000004,Smith and Sons Joinery Ltd,2015-05-01,2020-10-01
4,TW000005,Elm Street Plumbing Ltd,2021-02-10,NaN
5,TW000006,Quiet Fields Landscaping Ltd,2021-09-01,NaN
6,TW000007,Brightwater Electrical Ltd,2021-05-15,NaN
7,TW000008,Lambert Electrical Ltd,2016-03-01,NaN
8,TW000009,Smith Brothers Builders Ltd,2023-01-10,2024-05-01


**The label and the split.** A company *failed* if, within 24 months of being formed, it went into liquidation or administration or was dissolved (removed from the register). Companies are split by formation date: train on Jan–Sep 2021, validate on Oct–Dec 2021, test on 2022.

In [4]:
from src import evaluate

companies = mentions[mentions["kind"] == "company"].set_index("company_number").loc[queries]
labels = pd.DataFrame({
    "company_id": queries,
    "formed_on": companies["incorporated_on"].to_numpy(),
    "failed": evaluate.label_failures(mentions).loc[queries].to_numpy(),
    "split": None,
}, columns=ev.LABEL_COLS)

# split() gives three tables: write each one's name next to its companies
for name, part in zip(["train", "validation", "test"], evaluate.split(companies)):
    labels.loc[labels["company_id"].isin(part.index), "split"] = name

assert dict(zip(labels["company_id"], labels["failed"])) == truth["failed"]   # the known answers
labels

,company_id,formed_on,failed,split
0,TW000002,2021-04-20,True,train
1,TW000003,2021-06-01,False,train
2,TW000005,2021-02-10,False,train
3,TW000006,2021-09-01,False,train
4,TW000007,2021-05-15,False,train


All five tiny companies were formed between January and September 2021, so they all land in train. The tiny world has no test set.

## Baseline
Three simple rules our method has to beat:
1. **flag_nothing:** never flag. Failures are rare, so this is right most of the time.
2. **same_name:** two directors with the same name are the same person. What someone does by hand.
3. **officer_links:** only the links Companies House already makes through its officer id. What a careful supplier sees on the register today.

A company is flagged when one of its directors is linked to a company that went insolvent in the three years before it was formed.

In [5]:
from src import baseline

baseline_decisions = baseline.run_all(queries, mentions, claims)
table = baseline_decisions.pivot(index="query", columns="method", values="decision")
table["really failed"] = labels.set_index("company_id")["failed"]
table

method,flag_nothing,officer_links,same_name,really failed
query,,,,
TW000002,CLEAR,CLEAR,FLAG,True
TW000003,CLEAR,CLEAR,FLAG,False
TW000005,CLEAR,CLEAR,FLAG,False
TW000006,CLEAR,CLEAR,CLEAR,False
TW000007,CLEAR,CLEAR,CLEAR,False


In [6]:
# Count each baseline's flags against what really happened.
# To do: use evaluate.score_methods() instead once it is on main (card #80).
flagged = baseline_decisions["decision"] == "FLAG"
failed = baseline_decisions["query"].map(labels.set_index("company_id")["failed"])
counts = pd.DataFrame({"flags": flagged, "right flags": flagged & failed,
                       "false flags": flagged & ~failed, "misses": failed & ~flagged})
counts.groupby(baseline_decisions["method"]).sum()

,flags,right flags,false flags,misses
method,,,,
flag_nothing,0,0,0,1
officer_links,0,0,0,1
same_name,3,1,2,0


**What the tiny world shows** (five companies and one real failure, so a check and not a result):
- `flag_nothing` and `officer_links` flag nothing, so both miss the phoenix. The register gave its director a new officer id, and the officer links can't see past that.
- `same_name` catches the phoenix, but it also flags two honest companies whose director shares a name with someone else.

Those two kinds of mistake, the missed phoenix and the false alarm on a shared name, are what our method is meant to avoid.

## Model
A company number goes in. For each of its directors we ask: is this the same person as someone who ran a company that went insolvent earlier? Each possible match gets a probability instead of a yes or no. From those we compute P(predecessor), then P(fail), then the decision.

**Who might be the same person?** Comparing every person record with every other is too many pairs, so a cheap rule proposes the candidates: two person records are a candidate pair when they have the same surname and first initial, or the same birth month and year and a similar surname.

Each candidate pair then gets the numbers the matcher will learn from: how similar the names are, and whether birth month and year, postcode district and nationality agree. A `_missing` column says when a value is unknown, so "unknown" is not mistaken for "different". The register's officer id is never one of these numbers: it is the label the matcher is trained on.

In [7]:
from src import resolve

candidates = resolve.candidate_pairs(mentions)
features = resolve.pair_features(candidates, mentions)

assert set(truth["same_person"]) <= set(zip(candidates["a"], candidates["b"]))   # blocking loses no true pair
people = (mentions["kind"] == "person").sum()
print(len(candidates), "candidate pairs out of", people * (people - 1) // 2, "possible")
features

7 candidate pairs out of 36 possible


,name_similarity,name_missing,birth_same,birth_missing,postcode_same,postcode_missing,nationality_same,nationality_missing
0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
5,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
6,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0


**Learning from the register's own links.** The matcher needs examples with known answers, and the register gives them. Two records with the same officer id are the same person (label 1). A candidate pair with different officer ids counts as two people (label 0). That second label is sometimes wrong: the register can give one person two officer ids, and the tiny world's phoenix is exactly that case.

A logistic regression then learns one weight per feature and gives every pair a number `p` between 0 and 1. Nothing is merged: every pair stays in the table with its `p`.

In [8]:
labelled = resolve.labelled_pairs(mentions)
print("pairs per label:", labelled["label"].value_counts().to_dict())

model = resolve.train_matcher(resolve.pair_features(labelled, mentions), labelled["label"], SEED)
scored = resolve.scored_pairs(model, labelled, mentions)

# The known answers next to the register's label, to see where the label is wrong.
scored["really the same person"] = [pair in truth["same_person"] for pair in zip(scored["a"], scored["b"])]
scored[["label", "p", "really the same person"]]

pairs per label: {0: 5, 1: 2}


,label,p,really the same person
0,1,0.441023,True
1,0,0.169243,False
2,0,0.441023,True
3,0,0.169243,False
4,0,0.169243,False
5,0,0.169243,False
6,1,0.441023,True


In [9]:
resolve.matcher_weights(model).round(2)

name_similarity        0.00
name_missing           0.00
birth_same             0.68
birth_missing          0.00
postcode_same          0.68
postcode_missing       0.00
nationality_same       0.00
nationality_missing    0.00
intercept             -1.59
dtype: float64

**What the tiny world shows** (seven hand-made pairs, so a check that the matcher runs, not a trained model):
- The three pairs that agree on birth month and postcode district get `p` = 0.44. The four look-alikes, which share only a name, get 0.17.
- The row with label 0 that is really the same person is the phoenix. The register's label says "two people", yet the matcher gives it the same `p` as the pairs the register does link, because its features look the same. That is the trail the officer links alone can't see.
- The weights: `birth_same` and `postcode_same` are positive, so agreeing on either makes "same person" more likely. All the others are 0 here, because every tiny pair has the same name and nationality and no value is missing, so those columns can't tell the pairs apart.

The real weights come with the real data (card #36). `p` is only called a probability once it passes the calibration check on validation (card #88).

**P(predecessor).** Each candidate match is a chance that a director is the same person as someone who ran an insolvent earlier company. The chance that at least one match is right is 1 − (1 − p₁)(1 − p₂)…, which assumes the matches are independent: whether one is right tells us nothing about another. That isn't quite true, because a director is only one person; Sunday's one-identity rule fixes it (ADR-004).

In [10]:
from src.decide import p_predecessor

# Two independent 50% matches: the chance both are wrong is 0.5 × 0.5, so at least one is right with 0.75
p_predecessor([0.5, 0.5])

0.75

**The decision.** A straight line turns P(predecessor) into P(fail). It is fitted on the training companies: its left end is the failure rate of companies with no insolvent predecessor, its right end the failure rate of companies that surely have one. A company in between gets a rate in between. Three costs then say what to do: a missed case costs 10, a wrong flag 2, asking 1. That gives CLEAR at or below 10%, FLAG at or above 50%, and NOT ENOUGH EVIDENCE in between. A company with no director records is NOT ENOUGH EVIDENCE too.

Each decision also names **the one match to check first**: the match that would change the decision if we knew the truth about it. To find it, each match in turn is treated as certainly wrong, then as certainly right, and the decision is recomputed.

> **Stand-ins.** Seven tiny-world pairs can't train a matcher worth using. So the decision layer below still runs on probabilities **made up by hand** from the tiny world's known answers: 0.9 for a true same-person pair, 0.05 for a look-alike. The failure line is fitted, but on those made-up probabilities and four toy companies. This shows that the decision layer works. It says nothing about how good our method is.

In [11]:
from src import decide

# STAND-INS, made up by hand until the matcher is trained on real data (#36).
pairs = pd.DataFrame(
    [(a, b, 0.9, None, "stand-in", None) for a, b in truth["same_person"]]
    + [(a, b, 0.05, None, "stand-in", None) for a, b in truth["look_alikes"]],
    columns=ev.PAIR_COLS)
costs = (10, 2, 1)    # the team's choice: a missed case, a wrong flag, asking

# Each company's candidate matches, and from them its P(predecessor).
matches = {query: decide.candidate_matches(query, mentions, claims, pairs) for query in queries}
p_predecessors = [decide.company_p_predecessor(matches[query]) for query in queries]

# The failure line, fitted on the training companies. Here that is all five: the tiny world has no other split.
failed = labels.set_index("company_id").loc[queries, "failed"]   # each company's own label, in the queries' order
line = decide.fit_failure_line(p_predecessors, failed)
print("failure rate without an insolvent predecessor:", round(line[0], 2))
print("failure rate with one:", round(line[1], 2))

our_decisions = pd.DataFrame([decide.decide(query, matches[query], line, costs) for query in queries],
                             columns=ev.DECISION_COLS)

assert dict(zip(our_decisions["query"], our_decisions["decision"])) == truth["decision"]   # the known answers
# What would change each decision: the earlier company behind the match to check, and the new decision.
our_decisions["to check"] = [None if one is None else
                             f"{one['predecessor']}: {'same' if one['if_same_person'] else 'not the same'} person -> {one['becomes']}"
                             for one in our_decisions["flip"]]
our_decisions[["query", "decision", "p_predecessor", "p_fail", "to check"]]

failure rate without an insolvent predecessor: 0.0
failure rate with one: 1.0


,query,decision,p_predecessor,p_fail,to check
0,TW000002,FLAG,0.90,0.90,TW000001: not the same person -> CLEAR
1,TW000003,CLEAR,0.05,0.05,TW000001: same person -> FLAG
2,TW000005,CLEAR,0.05,0.05,TW000004: same person -> FLAG
3,TW000006,NOT ENOUGH EVIDENCE,NaN,NaN,NaN
4,TW000007,CLEAR,0.00,0.00,NaN


**What the tiny world shows** (four toy companies with a P(predecessor), so a check that the fit runs, not a result): the fitted line runs from 0 to 1. The closest straight line through these four points ends a little below 0 and a little above 1, so both ends were cut to the range a rate can have. All it says is that the one company with a likely predecessor is the one that failed.

On the real training companies the two ends will be real failure rates (card #36). If the right end came out below the FLAG threshold of 50%, FLAG could never fire. That is checked on validation (card #94).

## Evaluation
_Ours next to the baselines on data the model never saw, and where it fails._

Not yet. So far only the baselines are counted, in the Baseline section above, and only on the tiny world: five companies, one of which failed. Our method joins the table once its probabilities come from the matcher instead of by hand (card #31). The numbers that count come from the 2022 test companies, run once at the end (card #45).

## Demo
One company number in, the result out. This is the tiny world's phoenix, with the stand-in probabilities from the Model section.

In [12]:
answer = our_decisions.set_index("query").loc["TW000002"]
print("(tiny world, stand-in probabilities: a pipeline check, not a result)")
print(answer["decision"])
print(answer["reason"])

(tiny world, stand-in probabilities: a pipeline check, not a result)
FLAG
TW000002 may share a director with TW000001, an earlier company that went insolvent (90% sure it's the same person): its 90% chance of failing within two years is high enough to flag. This becomes CLEAR if the director matched with TW000001 turns out not to be the same person.
